# Setup: S3, Glue, Athena

## Solution Architecture

![Solution Architecture](../docs/architecture/solution.png)

In [1]:
from importlib.metadata import version, PackageNotFoundError
try:
    _sm_ver = version("sagemaker")
except PackageNotFoundError:
    _sm_ver = "0"
if int(_sm_ver.split(".")[0]) != 2:
    %pip uninstall -q -y sagemaker sagemaker-core sagemaker-train sagemaker-serve sagemaker-mlops
    %pip install -q "sagemaker>=2.200,<3"
    print(f"Replaced sagemaker {_sm_ver} with v2 - restarting kernel; re-run the notebook from the top.")
    import IPython
    IPython.Application.instance().kernel.do_shutdown(restart=True)
else:
    print("sagemaker", _sm_ver)

sagemaker 2.257.7


In [2]:
import sys, pathlib, boto3
from botocore.exceptions import ClientError, NoCredentialsError
ROOT = pathlib.Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.pamap2.config import REGION, build_config, save_config

session = boto3.Session(region_name=REGION)
try:
    ident = session.client("sts").get_caller_identity()
except (NoCredentialsError, ClientError) as e:
    raise SystemExit(f"AWS authentication failed (check credentials / region {REGION}): {e}")
account_id = ident["Account"]
print("Authenticated as", ident["Arn"], "in", REGION)
cfg = build_config(account_id)

Authenticated as arn:aws:sts::887757489409:assumed-role/LabRole/SageMaker in us-east-1


In [3]:
try:
    import sagemaker
    exec_role = sagemaker.get_execution_role()
    if exec_role.split("/")[-1] != "LabRole":
        print("WARNING: execution role differs from LabRole:", exec_role.split("/")[-1])
    else:
        print("Execution role matches LabRole")
except Exception as e:
    print("Could not cross-check execution role:", e)
print("Using role ARN:", cfg["role_arn"])

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


/opt/conda/lib/python3.12/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Execution role matches LabRole
Using role ARN: arn:aws:iam::887757489409:role/LabRole


In [4]:
s3 = session.client("s3")
bucket = cfg["bucket"]
try:
    s3.head_bucket(Bucket=bucket)
    print("bucket exists, reusing:", bucket)
except ClientError as e:
    if e.response["Error"]["Code"] not in ("404", "NoSuchBucket", "NotFound"):
        raise
    try:
        s3.create_bucket(Bucket=bucket)  # us-east-1
        print("created bucket:", bucket)
    except ClientError as e2:
        if e2.response["Error"]["Code"] != "BucketAlreadyOwnedByYou":
            raise
        print("bucket exists, reusing:", bucket)
s3.put_public_access_block(Bucket=bucket, PublicAccessBlockConfiguration={
    "BlockPublicAcls": True, "IgnorePublicAcls": True,
    "BlockPublicPolicy": True, "RestrictPublicBuckets": True})
s3.put_bucket_encryption(Bucket=bucket, ServerSideEncryptionConfiguration={
    "Rules": [{"ApplyServerSideEncryptionByDefault": {"SSEAlgorithm": "AES256"}}]})
for prefix in cfg["s3_prefixes"].values():
    try:
        s3.head_object(Bucket=bucket, Key=prefix)
    except ClientError as e:
        if e.response["Error"]["Code"] not in ("404", "NoSuchKey", "NotFound"):
            raise
        s3.put_object(Bucket=bucket, Key=prefix)
        print("created prefix", prefix)
print("prefixes ready")

created bucket: pamap2-har-887757489409-us-east-1
created prefix raw/
created prefix curated/
created prefix athena-results/
created prefix features/
created prefix models/
created prefix monitoring/
created prefix features/splits/
prefixes ready


In [5]:
glue = session.client("glue")
try:
    glue.create_database(DatabaseInput={"Name": cfg["glue_database"], "Description": "PAMAP2 HAR data lake"})
    print("created Glue database", cfg["glue_database"])
except glue.exceptions.AlreadyExistsException:
    print("Glue database exists, reusing:", cfg["glue_database"])

created Glue database pamap2_har


In [6]:
athena = session.client("athena")
try:
    athena.create_work_group(
        Name=cfg["athena_workgroup"],
        Description="PAMAP2 HAR queries",
        Configuration={"ResultConfiguration": {
            "OutputLocation": f"s3://{bucket}/{cfg['s3_prefixes']['athena-results']}",
            "EncryptionConfiguration": {"EncryptionOption": "SSE_S3"}},
            "EnforceWorkGroupConfiguration": True})
    print("created workgroup", cfg["athena_workgroup"])
except ClientError as e:
    if e.response["Error"]["Code"] != "InvalidRequestException" or "already created" not in e.response["Error"]["Message"]:
        raise
    print("workgroup exists, reusing:", cfg["athena_workgroup"])

created workgroup pamap2-har-wg


In [7]:
path = save_config(cfg)
print("Wrote", path)

Wrote /home/sagemaker-user/Wearable-Sensor-Based-Sports-Activity-Recognition/config/project_config.json
